# ============================================================
# PENGGABUNGAN SEMUA FILE HASIL CRAWL TWEET HARVEST
# ============================================================
# Input  : Semua file CSV di folder raw_data (±96 file, ±6000 tweets)
# Output : dataset_mbg_raw_gabungan.csv (sebelum filter relevansi)
# Platform: Google Colab
# ============================================================

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# KONFIGURASI PATH
# ============================================================

RAW_DIR    = '/content/drive/MyDrive/skripsi_mbg/data/raw_data'
OUTPUT_DIR = '/content/drive/MyDrive/skripsi_mbg/data/raw_final'
OUTPUT_FILE = f'{OUTPUT_DIR}/dataset_mbg_raw_gabungan.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 65)
print("PENGGABUNGAN FILE HASIL CRAWL TWEET HARVEST")
print("=" * 65)
print(f"Input folder : {RAW_DIR}")
print(f"Output file  : {OUTPUT_FILE}")

PENGGABUNGAN FILE HASIL CRAWL TWEET HARVEST
Input folder : /content/drive/MyDrive/skripsi_mbg/data/raw_data
Output file  : /content/drive/MyDrive/skripsi_mbg/data/raw_final/dataset_mbg_raw_gabungan.csv


In [3]:
# ============================================================
# BAGIAN 1 — SCAN DAN BACA SEMUA FILE CSV
# ============================================================
"""
Tweet Harvest menyimpan output per keyword per bulan.
Contoh nama file: makan_bergizi_gratis_2025-01.csv
                  program_mbg_2025-03.csv

Setiap file berisi kolom standar Tweet Harvest:
  - id_str              : ID unik tweet (string)
  - full_text           : Teks lengkap tweet
  - created_at          : Tanggal & waktu tweet
  - username            : Username akun
  - user_id_str         : ID unik user
  - favorite_count      : Jumlah like
  - retweet_count       : Jumlah retweet
  - reply_count         : Jumlah reply
  - quote_count         : Jumlah quote
  - lang                : Bahasa tweet
  - location            : Lokasi user (sering kosong)
  - tweet_url           : URL ke tweet
  - image_url           : URL gambar (jika ada)
  - in_reply_to_screen_name : Jika reply ke akun lain
  - conversation_id_str : ID percakapan
"""

print("\n" + "=" * 65)
print("BAGIAN 1: SCAN FILE CSV")
print("=" * 65)

# Cari semua file CSV di raw_data (termasuk subfolder)
semua_file = glob.glob(f'{RAW_DIR}/**/*.csv', recursive=True)
semua_file += glob.glob(f'{RAW_DIR}/*.csv')
semua_file = list(set(semua_file))  # hapus duplikat path
semua_file = sorted(semua_file)

print(f"\nTotal file CSV ditemukan: {len(semua_file)}")

if len(semua_file) == 0:
    print(f"\n⚠️  TIDAK ADA FILE CSV di: {RAW_DIR}")
    print("Pastikan path folder sudah benar!")
    raise FileNotFoundError(f"Tidak ada CSV di {RAW_DIR}")

# Tampilkan daftar file per keyword
print("\nDaftar file (preview 20 pertama):")
for i, f in enumerate(semua_file[:20]):
    nama = os.path.basename(f)
    try:
        ukuran = os.path.getsize(f)
        print(f"  [{i+1:>3}] {nama:<55} ({ukuran/1024:.1f} KB)")
    except:
        print(f"  [{i+1:>3}] {nama}")

if len(semua_file) > 20:
    print(f"  ... dan {len(semua_file) - 20} file lainnya")



BAGIAN 1: SCAN FILE CSV

Total file CSV ditemukan: 120

Daftar file (preview 20 pertama):
  [  1] kw10_hashtag_mbg_2025-01-06_2025-02-01.csv              (33.7 KB)
  [  2] kw10_hashtag_mbg_2025-02-01_2025-03-01.csv              (13.5 KB)
  [  3] kw10_hashtag_mbg_2025-03-01_2025-04-01.csv              (13.5 KB)
  [  4] kw10_hashtag_mbg_2025-04-01_2025-05-01.csv              (2.5 KB)
  [  5] kw10_hashtag_mbg_2025-05-01_2025-06-01.csv              (6.0 KB)
  [  6] kw10_hashtag_mbg_2025-06-01_2025-07-01.csv              (6.3 KB)
  [  7] kw10_hashtag_mbg_2025-07-01_2025-08-01.csv              (7.4 KB)
  [  8] kw10_hashtag_mbg_2025-08-01_2025-09-01.csv              (37.9 KB)
  [  9] kw10_hashtag_mbg_2025-09-01_2025-10-01.csv              (33.6 KB)
  [ 10] kw10_hashtag_mbg_2025-10-01_2025-11-01.csv              (50.8 KB)
  [ 11] kw10_hashtag_mbg_2025-11-01_2025-12-01.csv              (40.1 KB)
  [ 12] kw10_hashtag_mbg_2025-12-01_2025-12-31.csv              (7.5 KB)
  [ 13] kw1_makan_bergizi_

In [4]:
# ============================================================
# BAGIAN 2 — BACA DAN GABUNGKAN SEMUA FILE
# ============================================================
"""
Baca setiap file CSV dan gabungkan menjadi satu DataFrame.
Tangani berbagai kemungkinan error:
- File kosong (0 baris)
- File corrupt / encoding berbeda
- Kolom yang tidak konsisten antar file
"""

print("\n" + "=" * 65)
print("BAGIAN 2: BACA DAN GABUNGKAN SEMUA FILE")
print("=" * 65)
print("\nProses membaca file...")

daftar_df   = []
file_sukses = 0
file_kosong = 0
file_error  = 0
total_raw   = 0

for i, filepath in enumerate(semua_file):
    nama_file = os.path.basename(filepath)

    try:
        # Coba baca dengan encoding utf-8 dulu
        try:
            df_temp = pd.read_csv(filepath, encoding='utf-8', low_memory=False)
        except UnicodeDecodeError:
            # Fallback ke encoding latin-1 jika utf-8 gagal
            df_temp = pd.read_csv(filepath, encoding='latin-1', low_memory=False)

        # Lewati file kosong (0 baris data)
        if len(df_temp) == 0:
            file_kosong += 1
            print(f"  ⚠️  [{i+1:>3}] KOSONG    : {nama_file}")
            continue

        # Tambahkan info sumber file untuk traceability
        df_temp['_source_file'] = nama_file

        total_raw  += len(df_temp)
        file_sukses += 1
        daftar_df.append(df_temp)

        # Progress setiap 10 file
        if (i + 1) % 10 == 0:
            print(f"  ✅  [{i+1:>3}/{len(semua_file)}] Dibaca: {file_sukses} file, "
                  f"{total_raw:,} baris sejauh ini...")

    except Exception as e:
        file_error += 1
        print(f"  ❌  [{i+1:>3}] ERROR     : {nama_file} → {str(e)[:60]}")
        continue

print(f"\n{'─'*65}")
print(f"  File berhasil dibaca : {file_sukses}")
print(f"  File kosong (skip)   : {file_kosong}")
print(f"  File error (skip)    : {file_error}")
print(f"  Total baris raw      : {total_raw:,}")
print(f"{'─'*65}")

if len(daftar_df) == 0:
    raise ValueError("Tidak ada data yang berhasil dibaca!")

# Gabungkan semua DataFrame
print(f"\nMenggabungkan {len(daftar_df)} file...")
df_gabungan = pd.concat(daftar_df, ignore_index=True)
print(f"✅ Penggabungan selesai → shape: {df_gabungan.shape}")


BAGIAN 2: BACA DAN GABUNGKAN SEMUA FILE

Proses membaca file...
  ✅  [ 10/120] Dibaca: 10 file, 556 baris sejauh ini...
  ✅  [ 20/120] Dibaca: 20 file, 1,027 baris sejauh ini...
  ✅  [ 30/120] Dibaca: 30 file, 1,511 baris sejauh ini...
  ✅  [ 40/120] Dibaca: 40 file, 1,971 baris sejauh ini...
  ✅  [ 50/120] Dibaca: 50 file, 2,453 baris sejauh ini...
  ✅  [ 60/120] Dibaca: 60 file, 3,118 baris sejauh ini...
  ✅  [ 70/120] Dibaca: 70 file, 3,654 baris sejauh ini...
  ✅  [ 80/120] Dibaca: 80 file, 4,306 baris sejauh ini...
  ✅  [ 90/120] Dibaca: 90 file, 4,731 baris sejauh ini...
  ✅  [100/120] Dibaca: 100 file, 5,248 baris sejauh ini...
  ✅  [110/120] Dibaca: 110 file, 6,026 baris sejauh ini...
  ✅  [120/120] Dibaca: 120 file, 6,693 baris sejauh ini...

─────────────────────────────────────────────────────────────────
  File berhasil dibaca : 120
  File kosong (skip)   : 0
  File error (skip)    : 0
  Total baris raw      : 6,693
─────────────────────────────────────────────────────────

In [5]:
# ============================================================
# BAGIAN 3 — IDENTIFIKASI KOLOM
# ============================================================
"""
Tweet Harvest menghasilkan kolom yang konsisten, tapi
ada kemungkinan variasi nama kolom antar versi.
Identifikasi kolom ID dan teks yang tersedia.
"""

print("\n" + "=" * 65)
print("BAGIAN 3: IDENTIFIKASI KOLOM")
print("=" * 65)

print(f"\nSemua kolom yang tersedia ({len(df_gabungan.columns)}):")
for col in df_gabungan.columns:
    non_null = df_gabungan[col].notna().sum()
    pct      = non_null / len(df_gabungan) * 100
    print(f"  {col:<40} : {non_null:>6,} non-null ({pct:.1f}%)")

# Deteksi kolom ID tweet (untuk deduplication)
kandidat_id = ['id_str', 'tweet_id', 'id', 'Id', 'ID']
ID_COL = None
for k in kandidat_id:
    if k in df_gabungan.columns:
        ID_COL = k
        break

# Deteksi kolom teks utama
kandidat_text = ['full_text', 'text', 'tweet_text', 'content']
TEXT_COL = None
for k in kandidat_text:
    if k in df_gabungan.columns:
        TEXT_COL = k
        break

# Deteksi kolom tanggal
kandidat_date = ['created_at', 'date', 'timestamp', 'datetime']
DATE_COL = None
for k in kandidat_date:
    if k in df_gabungan.columns:
        DATE_COL = k
        break

print(f"\n{'─'*65}")
print(f"  Kolom ID    terdeteksi : {ID_COL   or '⚠️ TIDAK DITEMUKAN'}")
print(f"  Kolom TEXT  terdeteksi : {TEXT_COL or '⚠️ TIDAK DITEMUKAN'}")
print(f"  Kolom DATE  terdeteksi : {DATE_COL or '⚠️ TIDAK DITEMUKAN'}")
print(f"{'─'*65}")

if not ID_COL:
    print("\n⚠️  Kolom ID tidak ditemukan! Deduplication menggunakan teks.")
if not TEXT_COL:
    raise ValueError("Kolom teks tidak ditemukan! Cek nama kolom di file CSV.")



BAGIAN 3: IDENTIFIKASI KOLOM

Semua kolom yang tersedia (16):
  conversation_id_str                      :  6,693 non-null (100.0%)
  created_at                               :  6,693 non-null (100.0%)
  favorite_count                           :  6,693 non-null (100.0%)
  full_text                                :  6,693 non-null (100.0%)
  id_str                                   :  6,693 non-null (100.0%)
  image_url                                :  2,579 non-null (38.5%)
  in_reply_to_screen_name                  :  2,854 non-null (42.6%)
  lang                                     :  6,693 non-null (100.0%)
  location                                 :      0 non-null (0.0%)
  quote_count                              :  6,693 non-null (100.0%)
  reply_count                              :  6,693 non-null (100.0%)
  retweet_count                            :  6,693 non-null (100.0%)
  tweet_url                                :  6,693 non-null (100.0%)
  user_id_str                  

In [6]:
# ============================================================
# BAGIAN 4 — DEDUPLIKASI
# ============================================================
"""
Hapus tweet yang muncul lebih dari satu kali.
Tweet yang sama sering tertangkap oleh beberapa keyword berbeda
di bulan yang sama — ini penyebab utama raw >> clean.

Prioritas dedup:
1. Berdasarkan id_str (paling akurat)
2. Berdasarkan full_text (fallback jika tidak ada ID)
"""

print("\n" + "=" * 65)
print("BAGIAN 4: DEDUPLIKASI")
print("=" * 65)

jumlah_sebelum = len(df_gabungan)
print(f"\nJumlah sebelum dedup : {jumlah_sebelum:,}")

if ID_COL:
    # Dedup berdasarkan ID tweet (cara paling akurat)
    df_gabungan[ID_COL] = df_gabungan[ID_COL].astype(str).str.strip()
    df_dedup = df_gabungan.drop_duplicates(subset=[ID_COL], keep='first')
    print(f"Metode dedup         : Berdasarkan kolom '{ID_COL}'")
else:
    # Fallback: dedup berdasarkan teks
    df_gabungan[TEXT_COL] = df_gabungan[TEXT_COL].astype(str).str.strip()
    df_dedup = df_gabungan.drop_duplicates(subset=[TEXT_COL], keep='first')
    print(f"Metode dedup         : Berdasarkan kolom '{TEXT_COL}'")

df_dedup = df_dedup.reset_index(drop=True)

jumlah_sesudah = len(df_dedup)
duplikat_dihapus = jumlah_sebelum - jumlah_sesudah

print(f"Jumlah sesudah dedup : {jumlah_sesudah:,}")
print(f"Duplikat dihapus     : {duplikat_dihapus:,} "
      f"({duplikat_dihapus/jumlah_sebelum*100:.1f}% dari total raw)")



BAGIAN 4: DEDUPLIKASI

Jumlah sebelum dedup : 6,693
Metode dedup         : Berdasarkan kolom 'id_str'
Jumlah sesudah dedup : 1,554
Duplikat dihapus     : 5,139 (76.8% dari total raw)


In [ ]:

# ============================================================
# BAGIAN 5 — HAPUS TWEET KOSONG / TIDAK VALID
# ============================================================
"""
Hapus baris yang tidak memiliki teks atau teks tidak valid.
Ini berbeda dari deduplikasi — ini membersihkan data
yang memang tidak bisa digunakan untuk analisis sentimen.
"""

print("\n" + "=" * 65)
print("BAGIAN 5: HAPUS BARIS TIDAK VALID")
print("=" * 65)

jumlah_sebelum_clean = len(df_dedup)

# Hapus baris dengan teks kosong / NaN
df_valid = df_dedup[df_dedup[TEXT_COL].notna()].copy()
df_valid = df_valid[df_valid[TEXT_COL].astype(str).str.strip() != '']
df_valid = df_valid[df_valid[TEXT_COL].astype(str).str.strip() != 'nan']
df_valid = df_valid.reset_index(drop=True)

teks_invalid = jumlah_sebelum_clean - len(df_valid)
print(f"\nTeks kosong/invalid dihapus : {teks_invalid}")
print(f"Sisa setelah pembersihan    : {len(df_valid):,}")

In [ ]:
# ============================================================
# BAGIAN 6 — PARSING TANGGAL DAN DISTRIBUSI BULANAN
# ============================================================
"""
Parse kolom tanggal untuk analisis distribusi per bulan.
Diperlukan untuk memverifikasi coverage Januari-Desember 2025.
"""

print("\n" + "=" * 65)
print("BAGIAN 6: PARSING TANGGAL DAN DISTRIBUSI")
print("=" * 65)

if DATE_COL:
    try:
        # Parse tanggal dengan toleransi format berbeda
        df_valid[DATE_COL] = pd.to_datetime(
            df_valid[DATE_COL],
            errors='coerce',
            utc=True
        )
        # Konversi ke WIB (UTC+7)
        df_valid['tanggal_wib'] = df_valid[DATE_COL].dt.tz_convert('Asia/Jakarta')
        df_valid['bulan']       = df_valid['tanggal_wib'].dt.to_period('M')
        df_valid['tahun_bulan'] = df_valid['tanggal_wib'].dt.strftime('%Y-%m')

        # Distribusi per bulan
        dist_bulan = df_valid['bulan'].value_counts().sort_index()

        print(f"\nDistribusi tweet per bulan:")
        print(f"{'─'*35}")
        total_valid = 0
        for bulan, jumlah in dist_bulan.items():
            bar = '█' * (jumlah // 10)
            print(f"  {str(bulan):<10} : {jumlah:>5} tweet  {bar}")
            total_valid += jumlah
        print(f"{'─'*35}")
        print(f"  {'TOTAL':<10} : {total_valid:>5} tweet")

        # Cek coverage
        bulan_ada = [str(b) for b in dist_bulan.index]
        bulan_target = [f'2025-{m:02d}' for m in range(1, 13)]
        bulan_kurang = [b for b in bulan_target if b not in bulan_ada]

        if bulan_kurang:
            print(f"\n⚠️  Bulan yang kosong/kurang: {bulan_kurang}")
        else:
            print(f"\n✅ Coverage Jan-Des 2025 lengkap!")

    except Exception as e:
        print(f"⚠️  Gagal parse tanggal: {e}")
        print("   Lanjut tanpa informasi bulan...")
else:
    print("⚠️  Kolom tanggal tidak ditemukan, skip analisis bulan")


In [ ]:
# ============================================================
# BAGIAN 7 — STATISTIK AKHIR DAN PREVIEW DATA
# ============================================================

print("\n" + "=" * 65)
print("BAGIAN 7: STATISTIK AKHIR")
print("=" * 65)

print(f"\nRingkasan proses penggabungan:")
print(f"{'─'*50}")
print(f"  [1] Total file CSV dibaca     : {file_sukses:>6} file")
print(f"  [2] Total baris raw gabungan  : {total_raw:>6,} tweet")
print(f"  [3] Setelah deduplikasi       : {jumlah_sesudah:>6,} tweet "
      f"(-{duplikat_dihapus:,})")
print(f"  [4] Setelah hapus invalid     : {len(df_valid):>6,} tweet "
      f"(-{teks_invalid})")
print(f"{'─'*50}")
print(f"  DATASET RAW FINAL             : {len(df_valid):>6,} tweet")
print(f"{'─'*50}")

# Preview 5 tweet pertama
print(f"\nPreview 5 tweet pertama:")
print(f"{'─'*65}")
for i in range(min(5, len(df_valid))):
    teks = str(df_valid[TEXT_COL].iloc[i])[:100]
    print(f"\n[{i+1}] {teks}...")
    if DATE_COL and 'tahun_bulan' in df_valid.columns:
        print(f"     Tanggal: {df_valid['tahun_bulan'].iloc[i]}")
    if ID_COL:
        print(f"     ID     : {df_valid[ID_COL].iloc[i]}")

In [ ]:
# ============================================================
# BAGIAN 8 — SIMPAN FILE GABUNGAN
# ============================================================
"""
Simpan 2 versi:
1. File LENGKAP dengan semua kolom (untuk referensi)
2. File RINGKAS dengan kolom penting saja (untuk preprocessing)

File ini adalah RAW gabungan SEBELUM filter relevansi.
Langkah filter relevansi ada di notebook preprocessing.
"""

print("\n" + "=" * 65)
print("BAGIAN 8: SIMPAN FILE")
print("=" * 65)

# --- File 1: Lengkap (semua kolom) ---
df_valid.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
ukuran_mb = os.path.getsize(OUTPUT_FILE) / (1024*1024)
print(f"\n✅ File LENGKAP tersimpan:")
print(f"   Path  : {OUTPUT_FILE}")
print(f"   Ukuran: {ukuran_mb:.2f} MB")
print(f"   Baris : {len(df_valid):,}")
print(f"   Kolom : {len(df_valid.columns)}")

# --- File 2: Ringkas (kolom penting saja) ---
# Pilih kolom yang ada
kolom_penting = []
for col in [ID_COL, TEXT_COL, DATE_COL,
            'username', 'lang', 'favorite_count',
            'retweet_count', 'reply_count', 'tweet_url',
            'tahun_bulan', '_source_file']:
    if col and col in df_valid.columns:
        kolom_penting.append(col)

df_ringkas = df_valid[kolom_penting].copy()

output_ringkas = OUTPUT_FILE.replace('.csv', '_ringkas.csv')
df_ringkas.to_csv(output_ringkas, index=False, encoding='utf-8')
ukuran_ringkas = os.path.getsize(output_ringkas) / (1024*1024)
print(f"\n✅ File RINGKAS tersimpan:")
print(f"   Path  : {output_ringkas}")
print(f"   Ukuran: {ukuran_ringkas:.2f} MB")
print(f"   Baris : {len(df_ringkas):,}")
print(f"   Kolom : {kolom_penting}")

In [ ]:
# ============================================================
# BAGIAN 9 — ANALISIS SUMBER FILE (OPSIONAL)
# ============================================================
"""
Analisis berapa tweet yang berasal dari masing-masing keyword.
Berguna untuk laporan metodologi di BAB III skripsi.
"""

print("\n" + "=" * 65)
print("BAGIAN 9: ANALISIS PER KEYWORD (SUMBER FILE)")
print("=" * 65)

if '_source_file' in df_valid.columns:
    # Ekstrak nama keyword dari nama file
    # Format Tweet Harvest: keyword_YYYY-MM.csv
    df_valid['keyword_source'] = df_valid['_source_file'].apply(
        lambda x: '_'.join(str(x).replace('.csv', '').split('_')[:-1])
        if '_' in str(x) else str(x).replace('.csv', '')
    )

    dist_keyword = df_valid['keyword_source'].value_counts()

    print(f"\nDistribusi tweet per keyword:")
    print(f"{'─'*60}")
    for kw, jumlah in dist_keyword.items():
        pct = jumlah / len(df_valid) * 100
        bar = '█' * int(pct / 2)
        print(f"  {kw[:40]:<42}: {jumlah:>5} ({pct:>5.1f}%) {bar}")
    print(f"{'─'*60}")

print("\n" + "=" * 65)
print("✅ PENGGABUNGAN SELESAI!")
print("=" * 65)
print(f"\nFile output:")
print(f"  1. dataset_mbg_raw_gabungan.csv        ← semua kolom")
print(f"  2. dataset_mbg_raw_gabungan_ringkas.csv ← kolom penting saja")
print(f"\nPath: /content/drive/MyDrive/skripsi_mbg/data/raw/")
print(f"\nLangkah selanjutnya:")
print(f"  → Jalankan notebook preprocessing untuk filter relevansi")
print(f"     dan pembersihan teks")